#### to compare sleep staging results between icu and sleeplab, we want to have sleep stage predictions for sleeplab files too. here, we do not use the 'final' model but the model according to the fold_id. (this is the data the airgo sleep staging models were trained on.)

In [1]:
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import sys

sys.path.append('/home/wolfgang/repos/ICU-Sleep')
from airgo_features import airgo_actigraphy_features

sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file

sys.path.append('/home/wolfgang/repos/AirGoSleepPyT0')
from segment_signal import *
from collections import Counter
from scipy import io as sio

import torch as th
th.cuda.empty_cache()
from utils import softmax
from braindecode.torch_ext.util import np_to_var, var_to_np

from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

In [2]:
def clip_z_normalize(signal):

    signal_clipped = np.clip(signal.dropna(), np.percentile(signal.dropna(),1), np.percentile(signal.dropna(),99))
    mean = np.mean(signal_clipped.astype(np.float32))
    std = np.std(signal_clipped.astype(np.float32))
    signal = (signal - mean)/std

    return signal

 
def myprint(seg_mask):
    sm = Counter([x.split('_')[0] for x in seg_mask])
    for ex in seg_mask_explanation:
        if ex in sm:
            print(('%s: %d/%d, %g%%'%(ex,sm[ex],len(seg_mask),sm[ex]*100./len(seg_mask))))
    

In [3]:
def segment_signal_only(signal, window_time, step_time, fs, 
                        notch_freq=None, bandpass_freq=None, start_end_remove_window_num=1, 
                        amplitude_thres=None, amplitude_thres_low=None, n_jobs=-1, 
                        to_remove_mean=False, 
                        std_thres1=0.00001, std_thres2=0.00001):

    # segmentation:

    window_size = int(round(window_time*fs))
    step_size = int(round(step_time*fs))

    # prepare segmentation:
    start_ids = np.arange(0, signal.shape[1]-window_size+1, step_size)
    if start_end_remove_window_num>0:
        start_ids = start_ids[start_end_remove_window_num:-start_end_remove_window_num]
    seg_masks = [seg_mask_explanation[0]]*len(start_ids)

    ### check for high amplitude (airgo not worn) before detrending signal:
    if amplitude_thres is not None:
    ## find large amplitude in signal
        signal_segs_temp = signal[:,[np.arange(x,x+window_size) for x in start_ids]].transpose(1,0,2)  # (#window, #ch, window_size+2padding)

        amplitude_large2d = np.any(np.abs(signal_segs_temp)>amplitude_thres, axis=2)
        amplitude_large1d = np.where(np.any(amplitude_large2d, axis=1))[0]
        for i in amplitude_large1d:
            seg_masks[i] = '%s'%(seg_mask_explanation[4])

    if amplitude_thres_low is not None:
        ## find low amplitude in signal
        signal_segs_temp = signal[:,[np.arange(x,x+window_size) for x in start_ids]].transpose(1,0,2)  # (#window, #ch, window_size+2padding)
        amplitude_low2d = np.any(np.abs(signal_segs_temp)<amplitude_thres_low, axis=2)
        amplitude_low1d = np.where(np.any(amplitude_low2d, axis=1))[0]
        for i in amplitude_low1d:
            seg_masks[i] = '%s'%(seg_mask_explanation[4])

    # normalize signal:
    signal = clip_z_normalize(pd.Series(signal[0,:]))
    signal = signal[np.newaxis, :]

    # create signal segments:
    signal_segs = signal[:,[np.arange(x,x+window_size) for x in start_ids]].transpose(1,0,2)  # (#window, #ch, window_size+2padding)

    ## find nan in signal segs

    nan2d = np.any(np.isnan(signal_segs), axis=2)
    nan1d = np.where(np.any(nan2d, axis=1))[0]
    for i in nan1d:
        seg_masks[i] = '%s'%(seg_mask_explanation[3])

    return signal_segs, seg_masks, start_ids

In [4]:

seg_mask_explanation = [
    'normal',
    'around sleep stage change point',
    'NaN in sleep stage',
    'NaN in signal',
    'overly high/low amplitude',
    'flat signal',
    'NaN in feature',
    'NaN in spectrum',
    'muscle artifact',
    'spurious spectrum',
    'missing or additional R peak']


In [5]:
# first time, use just 10Hz version, then append to same data, i.e. give each sleep staging 
# result a name/id and if more sleep stage models are applied, just more arrays with different ids are appended.

# input_dir = '/media/mad3/Projects/AirGo_PSG/10Hz_data'
input_dir = '/media/ssd2/sleeplab_sleepstaging'
output_dir = '/media/ssd2/sleeplab_sleepstaging'

In [6]:
# step 1: load data

verbose = False
files = os.listdir(input_dir)
files.sort()
print(len(files))
print(files[:3])

window_time = 270
step_time =30
start_end_remove_window_num=1
n_jobs=-1
to_remove_mean=False


412
['psg_airgo_10hz_001.h5', 'psg_airgo_10hz_002.h5', 'psg_airgo_10hz_003.h5']


In [7]:
# add sleep stage prediction #1, 06/22/2020:
# model_id = 'amendsumeffort'
# input_signal = 'band_smooth'
# models_dir = '/media/mad3/Projects/AirGoSleepStaging/final_models/AirGoFilter_AmendSumEffort'

# add sleep stage prediction #2, 06/24/2020:
model_id = 'activity1sec'
input_signal = 'acc_ai_1sec'
models_dir = '/media/mad3/Projects/AirGoSleepStaging/final_models/ActivityIndex1sec'

In [8]:
def cnn_and_lstm_predict(X, cnn_model, lstm_model):

    with th.no_grad():
        output, H = cnn_model(X)
        output = var_to_np(output)
        H = var_to_np(H)

        H = np.expand_dims(H, axis=0)
        H = np_to_var(H)
        H = H.cuda()

        yp = lstm_model(H)[0]
        yp = var_to_np(yp)
        yp = np.squeeze(yp)

        yp = softmax(yp)

    th.cuda.empty_cache()
    
    del H, cnn_model, lstm_model, output
    
    return yp


In [9]:
# GET FOLD INFOS:
fold_dir = '/home/wolfgang/repos/AirGoSleepPyT0/fold_ids/'

if 0:
    
    folds = pd.DataFrame()

    for fold_id in range(5):
        fold_tmp = pd.read_csv(os.path.join(fold_dir, f'test_files_fold{fold_id}.csv'))
        fold_tmp['fold_id'] = fold_id
        folds = pd.concat([folds, fold_tmp])

    folds.to_csv(os.path.join(fold_dir, f'all_test_folds.csv'), index=False)
    
else:
    folds = pd.read_csv(os.path.join(fold_dir, f'all_test_folds.csv'))
    
folds.fileID = [x.replace('Feature_Air','psg_airgo_10hz_').replace('.mat','.h5') for x in folds.fileID]
print(folds.head(3))

                  fileID  fold_id
0  psg_airgo_10hz_002.h5        0
1  psg_airgo_10hz_013.h5        0
2  psg_airgo_10hz_025.h5        0


In [13]:
for fold_id in range(5):
#     fold_id = 0
    print(f'Fold {fold_id}')
    # files for fold
    files = folds.loc[folds.fold_id == fold_id, 'fileID']

    # models for fold:
    cnn_model = os.path.join(models_dir, f'fold_{fold_id}', 'CNN_best_model.pth')
    cnn_model = th.load(cnn_model);
    cnn_model.eval();

    lstm_model = os.path.join(models_dir, f'fold_{fold_id}', 'RNN_abest_model.pth')
    lstm_model = th.load(lstm_model);
    lstm_model.eval();

    cnn_lstm_prediction_routine()

Fold 0


100%|██████████| 82/82 [04:30<00:00,  3.30s/it]


Fold 1


100%|██████████| 84/84 [04:46<00:00,  3.41s/it]


Fold 2


100%|██████████| 82/82 [04:36<00:00,  3.37s/it]


Fold 3


100%|██████████| 84/84 [04:27<00:00,  3.19s/it]


Fold 4


100%|██████████| 80/80 [04:22<00:00,  3.28s/it]


In [12]:

def cnn_lstm_prediction_routine():

    for file_tmp in tqdm(files):
    #     try:
        if 1:

    #         if os.path.exists(os.path.join(output_dir, file_tmp)):
    #             continue

            data, hdr = load_sleep_data(os.path.join(input_dir, file_tmp), idx_to_datetime=1)
            fs = hdr['fs']
            data.columns = [x.lower() for x in data.columns]

            hdr['study_id'] = np.int32(hdr['study_id'])
            hdr['MRN'] = np.int32(hdr['MRN'])
            hdr['fs'] = np.int32(hdr['fs'])
            hdr['start_datetime'] = np.array([hdr['start_datetime'].year,hdr['start_datetime'].month,hdr['start_datetime'].day, hdr['start_datetime'].hour, hdr['start_datetime'].minute, hdr['start_datetime'].second, hdr['start_datetime'].microsecond])

            if not 'band' in data.columns:
                print('No AirGo data at all.')
                continue

            # print(data.band.dropna().shape[0])
            if data.band.dropna().shape[0] == 0:
                print('No AirGo data, seems like wrong category assigned')
                continue

            data.rename({'spo2%':'spo2'}, axis=1, inplace=True)


    #         if not 'acc_ai_1sec' in data.columns:

    #             features_to_keep = ['art1d', 'art1s', 'hr', 'spo2', 
    #                             'accx', 'accy', 'accz', 'band']
    #             features_to_keep = [x for x in features_to_keep if x in data.columns]
    #             data = data[features_to_keep]


    #             # also compute actigraphy activity:
    #             data = airgo_actigraphy_features(data, fs=10)
    #             data.drop(['accx', 'accy', 'accz'], axis=1, inplace=True)

            data['band_smooth'] = data['band'].reset_index()['band'].rolling(int(0.5*fs), center=True, min_periods=1).mean().values

            if verbose:
                print_data_availability(data.band_smooth)

            if input_signal == 'band_smooth':
                signal = data.band_smooth.values
                signal = signal[np.newaxis, :]

                amplitude_thres = 4075
                amplitude_thres_low = 400 # only in use for AirGo.
                std_thres1=0.00001
                std_thres2=0.00001

                if any(pd.Series(signal[0,:]).dropna()>5000):
                    print('airgo version pre V5, remove amplitude thresholds')
                    amplitude_thres = 100000000
                    amplitude_thres_low = 0 # only in use for AirGo.

            elif input_signal == 'acc_ai_1sec':
                signal = data.band_smooth.values
                signal = signal[np.newaxis, :]
                amplitude_thres = None
                amplitude_thres_low = None
                std_thres1=0
                std_thres2=0

            if any(pd.Series(signal[0,:]).dropna()>5000):
                print('airgo version pre V5, remove amplitude thresholds')
                amplitude_thres = 100000000
                amplitude_thres_low = 0 # only in use for AirGo.

            segs, seg_mask, seg_start_pos = segment_signal_only(signal, window_time, step_time, fs, 
                                    start_end_remove_window_num=start_end_remove_window_num, 
                                    amplitude_thres=amplitude_thres, amplitude_thres_low=amplitude_thres_low,
                                n_jobs=n_jobs, to_remove_mean=to_remove_mean)

            normal_only = True
            if normal_only:
                good_ids = np.where(np.in1d(seg_mask,seg_mask_explanation[:2]))[0]
                if len(good_ids)<=0:
                    myprint(seg_mask)
                    raise ValueError('No normal segments')
                segs = segs[good_ids]
                seg_start_pos = seg_start_pos[good_ids]
            else:
                good_ids = np.arange(len(seg_mask))

            # REMOVE INF and NAN if still here...
            # segs.shape
            isgood = np.isfinite(segs).all(axis=2).flatten()
            segs = segs[isgood,:,:]
            seg_start_pos = seg_start_pos[isgood]

            X = th.tensor(segs).float()

            yp = cnn_and_lstm_predict(X, cnn_model, lstm_model)

            yp = np.argmax(yp, axis=1) + 1

            data[f'stage_pred_{model_id}'] = np.nan
            stage_loc = data.iloc[seg_start_pos].index
            data.loc[stage_loc, f'stage_pred_{model_id}'] = yp
            data[f'stage_pred_{model_id}'].fillna(method='pad',
                                                  limit=30*fs-1, inplace=True)
            write_to_hdf5_file(data, os.path.join(output_dir, file_tmp), 
                               hdr=hdr, overwrite=True)

    #         del data, X, yp, segs
            th.cuda.empty_cache()

    #     except Exception as e:
    #         print(file_tmp)
    #         print(e)
    #     myprint(seg_mask)


In [19]:
files_done = os.listdir(output_dir)
file_tmp = files_done[0]
data, hdr = load_sleep_data(os.path.join(output_dir, file_tmp), idx_to_datetime=1)

In [22]:
data.stage_pred.dropna()

2019-02-23 22:30:34.200    5.0
2019-02-23 22:30:34.300    5.0
2019-02-23 22:30:34.400    5.0
2019-02-23 22:30:34.500    5.0
2019-02-23 22:30:34.600    5.0
                          ... 
2019-02-24 06:38:03.700    4.0
2019-02-24 06:38:03.800    4.0
2019-02-24 06:38:03.900    4.0
2019-02-24 06:38:04.000    4.0
2019-02-24 06:38:04.100    4.0
Freq: 100L, Name: stage_pred, Length: 292500, dtype: float32

In [33]:
plt.figure(figsize=(7,4))
plt.plot(data.stage, c='r')
plt.plot(data.stage_pred, c='b')
plt.yticks([1,2,3,4,5], ['N3','N2','N1','R','W'])
plt.ylim([0.5,5.5])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

(0.5, 5.5)